In [1]:
import pandas as pd
import numpy as np
import os
from xgboost import XGBClassifier
import torch
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
import glob

In [2]:
def metrics(tn, fp, fn, tp):
    tn = tn.astype(np.float32)
    fp = fp.astype(np.float32)
    fn = fn.astype(np.float32)
    tp = tp.astype(np.float32)

    cm_dict = {'predict\\actual':['Positive', 'Negative']
               ,'Positive':[tp, fn]
               ,'Negative':[fp, tn]}

    cm_df = pd.DataFrame(cm_dict)
    
    accuracy = round((tp + tn) / (tp + tn + fp + fn), 4) if (tp + tn + fp + fn) > 0 else 0  
    sensitivity = round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0
    specificity = round(tn / (tn + fp), 4) if (tn + fp) > 0 else 0
    ppv = round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0
    npv = round(tn / (tn + fn), 4) if (tn + fn) > 0 else 0
    mcc = round((tp * tn - fp * fn) / np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)), 4) if (tp + fp) > 0 and (tp + fn) > 0 and (tn + fp) > 0 and (tn + fn) > 0 else 0

    metric_dict = {'metric':['Accuruacy', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'MCC'],
                   'Value':[accuracy, sensitivity, specificity, ppv, npv, mcc]}

    metric_df = pd.DataFrame(metric_dict)

    return cm_df, metric_df

In [ ]:
per_window_dir = r"D:\M143020071\raw_data_result\SKNA_signal\ch1\sr10000_500_2000_15s_ECG_signal_rpeak_5_10min_longer_1000pts_mr_9win/"
df_h = pd.read_csv(os.path.join(per_window_dir, "non_mace_windows_info.csv"))
df_p=pd.read_csv(os.path.join(per_window_dir, "mace_windows_info.csv"))

data_dir_mace=r"D:\M143020071\raw_data_result\SKNA_signal\ch1\sr10000_500_2000_15s_ECG_signal_rpeak_5_10min_longer_1000pts_mr_9win\mace_zscore/"
data_dir_non_mace=r"D:\M143020071\raw_data_result\SKNA_signal\ch1\sr10000_500_2000_15s_ECG_signal_rpeak_5_10min_longer_1000pts_mr_9win\non_mace_zscore/"

save_data_path = r"D:\M143020071\xgboost_result\static_dynamic_ECGFounder_askna_mr_feature_result/"
save_combine_data_path = save_data_path + "combine/"
print("success")

success


In [4]:
print("開始處理 MACE 資料...")
mace_list = [] 
mace_files = glob.glob(os.path.join(data_dir_mace, "*.npy"))

for file_path in mace_files:
    data = np.load(file_path)
    mace_list.append(data)

if len(mace_list) > 0:
    mace_stacked = np.vstack(mace_list)
    print(f"MACE 資料處理完成。疊加後的形狀為: {mace_stacked.shape}")
else:
    print("未在 MACE 資料夾中找到 .npy 檔案")

print("開始處理 Non-MACE 資料...")
non_mace_list = [] 
non_mace_files = glob.glob(os.path.join(data_dir_non_mace, "*.npy"))

for file_path in non_mace_files:
    data = np.load(file_path)
    non_mace_list.append(data)

if len(non_mace_list) > 0:
    non_mace_stacked = np.vstack(non_mace_list)
    print(f"Non-MACE 資料處理完成。疊加後的形狀為: {non_mace_stacked.shape}") #432人*9特徵
else:
    print("未在 Non-MACE 資料夾中找到 .npy 檔案")

print("處理結束。")

開始處理 MACE 資料...
MACE 資料處理完成。疊加後的形狀為: (810, 151)
開始處理 Non-MACE 資料...
Non-MACE 資料處理完成。疊加後的形狀為: (3888, 151)
處理結束。


In [5]:
id_p = np.array([os.path.basename(f).replace('.npy', '') for f in mace_files])
id_h = np.array([os.path.basename(f).replace('.npy', '') for f in non_mace_files])
print(f"id_p (MACE ID) 範例: {id_p[:3]}")
print(f"id_h (Non-MACE ID) 範例: {id_h[:3]}")

p = mace_stacked

h = non_mace_stacked

print("-" * 30)
print(f"p 的形狀: {p.shape}")
print(f"h 的形狀: {h.shape}")


id_p (MACE ID) 範例: ['MACE021' 'MACE028' 'MACE033']
id_h (Non-MACE ID) 範例: ['MACE001' 'MACE002' 'MACE003']
------------------------------
p 的形狀: (810, 151)
h 的形狀: (3888, 151)


In [6]:
df_h = df_h.rename(columns={'ID': 'file_name', 'Windows': 'window_per_person'})
df_p = df_p.rename(columns={'ID': 'file_name', 'Windows': 'window_per_person'})

h_file_list = df_h['file_name'].values
p_file_list = df_p['file_name'].values

h_window_per_person = df_h[['file_name', 'window_per_person']].values
p_window_per_person = df_p[['file_name', 'window_per_person']].values
print(h_window_per_person[0:10])
print("-----------")
print(p_window_per_person[0:10])
window_per_person = pd.concat([df_h, df_p], axis=0)


df_h[['file_name', 'window_per_person']].to_csv(os.path.join(save_data_path, "h_window_per_person.csv"), index=False)

# 儲存 Patient 的視窗表
df_p[['file_name', 'window_per_person']].to_csv(os.path.join(save_data_path, "p_window_per_person.csv"), index=False)
print(f"✅ 已成功儲存視窗對照表至: {save_data_path}")
print(f"h_file_list length: {len(h_file_list)}")
print(f"p_file_list length: {len(p_file_list)}")


[['MACE001' 9]
 ['MACE002' 9]
 ['MACE003' 9]
 ['MACE004' 9]
 ['MACE005' 9]
 ['MACE006' 9]
 ['MACE007' 9]
 ['MACE008' 9]
 ['MACE009' 9]
 ['MACE010' 9]]
-----------
[['MACE021' 9]
 ['MACE028' 9]
 ['MACE033' 9]
 ['MACE044' 9]
 ['MACE046' 9]
 ['MACE051' 9]
 ['MACE052' 9]
 ['MACE054' 9]
 ['MACE055' 9]
 ['MACE061' 9]]
✅ 已成功儲存視窗對照表至: D:\M143020071\xgboost_result\static_dynamic_ECGFounder_askna_mr_feature_result/
h_file_list length: 432
p_file_list length: 90


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

st_h = 0
st_p = 0
current_h_idx = 0
current_p_idx = 0

# 累加用的變數保留，為了最後計算整體指標
all_test_tp = 0
all_test_fp = 0
all_test_fn = 0
all_test_tn = 0
all_train_tp = 0
all_train_fp = 0
all_train_fn = 0
all_train_tn = 0

# ---------------------------------------------------------
# [修改] 這裡原本是定義 list 來存資料，現在不需要這些 list 了
# 因為我們會在迴圈內直接存檔，所以這些可以註解掉或移除
# all_results_y_train_label = []
# all_results_y_test_label = []
# all_results_yhat_train_probability = []
# all_results_yhat_test_probability = []
# ---------------------------------------------------------

# 確保輸出路徑存在 (如果沒有資料夾可能會報錯)
if not os.path.exists(save_data_path):
    os.makedirs(save_data_path)
    print(f"Created directory: {save_data_path}")

for i in tqdm(range((int((len(p_file_list))/1))), desc="iteration"):
    # 這裡保留原本的邏輯
    if i < len(h_file_list)-len(p_file_list)*4:
        h_size = 5
    else:
        h_size = 4

    p_size = 1

    ed_h_idx = current_h_idx + h_size
    ed_p_idx = current_p_idx + p_size

    ed_h = np.sum(h_window_per_person[current_h_idx : ed_h_idx, 1])
    ed_p = np.sum(p_window_per_person[current_p_idx : ed_p_idx, 1])

    test = np.vstack([h[st_h:st_h+ed_h, :], p[st_p:st_p+ed_p, :]])

    train_h = np.vstack([h[:st_h, :], h[st_h + ed_h:, :]])
    train_p = np.vstack([p[:st_p, :], p[st_p + ed_p:, :]])
    train = np.vstack([train_h, train_p])

    test_file_name = list(h_file_list[current_h_idx:ed_h_idx]) + list(p_file_list[current_p_idx:ed_p_idx])
    train_file_name = list(h_file_list[:current_h_idx]) + list(h_file_list[ed_h_idx:]) + list(p_file_list[:current_p_idx]) + list(p_file_list[ed_p_idx:])

    st_h += ed_h
    st_p += ed_p
    current_h_idx = ed_h_idx
    current_p_idx = ed_p_idx

    test_window_name = []
    for j, name in enumerate(test_file_name):
        mask = window_per_person['file_name'].isin([name])
        win_num = window_per_person[mask]['window_per_person'].reset_index(drop=True)
        win_num = int(win_num.iloc[0])
        test_window_name += [test_file_name[j]] * win_num
    test_file_name_df = pd.DataFrame(test_window_name, columns=['file_name'])

    train_window_name = []
    for j, name in enumerate(train_file_name):
        mask = window_per_person['file_name'].isin([name])
        win_num = window_per_person[mask]['window_per_person'].reset_index(drop=True)
        win_num = int(win_num.iloc[0])
        train_window_name += [train_file_name[j]] * win_num
    train_file_name_df = pd.DataFrame(train_window_name, columns=['file_name'])

    X_train = train[:, 1:].astype(np.float32)
    y_train = train[:, 0].reshape(-1).astype(np.int32)
    X_test = test[:, 1:].astype(np.float32)
    y_test = test[:, 0].reshape(-1).astype(np.int32)
    
    # 模型訓練
    clf = XGBClassifier(device=device)
    clf.fit(X_train, y_train)

    yhat_train = clf.predict(X_train)
    yhat_train_proba = clf.predict_proba(X_train)[:, 1]
    yhat_test = clf.predict(X_test)
    yhat_test_proba = clf.predict_proba(X_test)[:, 1]

    # 建立 DataFrame
    y_train_df = pd.DataFrame(y_train, columns=['y_train'])
    y_test_df = pd.DataFrame(y_test, columns=['y_test'])
    yhat_train_df = pd.DataFrame(yhat_train, columns=['yhat_train'])
    yhat_test_df = pd.DataFrame(yhat_test, columns=['yhat_test'])
    yhat_train_prob_df = pd.DataFrame(yhat_train_proba, columns=['yhat_train_proba'])
    yhat_test_prob_df = pd.DataFrame(yhat_test_proba, columns=['yhat_test_proba'])

    y_train_label = pd.concat([train_file_name_df.reset_index(drop=True), y_train_df.reset_index(drop=True), yhat_train_df], axis=1)
    y_test_label = pd.concat([test_file_name_df.reset_index(drop=True), y_test_df.reset_index(drop=True), yhat_test_df], axis=1)
    yhat_train_probability = pd.concat([train_file_name_df.reset_index(drop=True), yhat_train_prob_df], axis=1)
    yhat_test_probability = pd.concat([test_file_name_df.reset_index(drop=True), yhat_test_prob_df], axis=1)

    y_train_label['iteration'] = i 
    y_test_label['iteration'] = i
    yhat_train_probability['iteration'] = i
    yhat_test_probability['iteration'] = i

    # ==============================================================================
    # [重點修改] 這裡直接存檔，檔名加上 _i (例如 y_train_label_0.csv)
    # ==============================================================================
    y_train_label.to_csv(os.path.join(save_data_path, f"y_train_label_{i}.csv"), index=False)
    y_test_label.to_csv(os.path.join(save_data_path, f"y_test_label_{i}.csv"), index=False)
    yhat_train_probability.to_csv(os.path.join(save_data_path, f"yhat_train_probability_{i}.csv"), index=False)
    yhat_test_probability.to_csv(os.path.join(save_data_path, f"yhat_test_probability_{i}.csv"), index=False)
    
    # 移除原本的 append (省記憶體)
    # all_results_y_train_label.append(y_train_label)
    # all_results_y_test_label.append(y_test_label)
    # ...
    
    # 混淆矩陣累加 (保留這段，這樣最後才能印出總表)
    train_tn, train_fp, train_fn, train_tp = confusion_matrix(y_train, yhat_train).ravel()
    all_train_tp += train_tp
    all_train_fn += train_fn
    all_train_fp += train_fp
    all_train_tn += train_tn

    test_tn, test_fp, test_fn, test_tp = confusion_matrix(y_test, yhat_test).ravel()
    all_test_tp += test_tp
    all_test_fn += test_fn
    all_test_fp += test_fp
    all_test_tn += test_tn

# ----------------------------------------------------------------------------------------------#
# 迴圈結束後，因為我們沒有用 list 存資料了，所以原本那段 concat 的程式碼要註解掉
# final_y_train_label = pd.concat(all_results_y_train_label, ...)
# final_y_train_label.to_csv(...) 
# (這些都已經在迴圈內存完了，所以不用再執行)
# ----------------------------------------------------------------------------------------------#

# 計算並儲存最後的 Metric 結果 (這段保留，因為我們有累加 TP/FP/FN/TN)
if not os.path.exists(save_data_path):
    os.makedirs(save_data_path)

train_cm_df, train_metric_df = metrics(all_train_tn, all_train_fp, all_train_fn, all_train_tp)
test_cm_df, test_metric_df = metrics(all_test_tn, all_test_fp, all_test_fn, all_test_tp)

train_cm_df.to_csv(save_data_path + f"train_cm.csv", index=False)
train_metric_df.to_csv(save_data_path + f"train_metric.csv", index=False)
test_cm_df.to_csv(save_data_path + f"test_cm.csv", index=False)
test_metric_df.to_csv(save_data_path + f"test_metric.csv", index=False)

print('===========   TRAIN RESULTS   ===========')
print(f"confusion matrix of train :\n{train_cm_df.to_string(index=False)}\n")
print(f"metric of train :\n{train_metric_df.to_string(index=False)}\n")

print('===========   TEST RESULTS   ===========')
print(f"confusion matrix of test :\n{test_cm_df.to_string(index=False)}\n")
print(f"metric of test :\n{test_metric_df.to_string(index=False)}\n")

cuda


iteration:   0%|          | 0/90 [00:00<?, ?it/s]c:\Users\M143020071\miniconda3\envs\mace-env\lib\site-packages\xgboost\core.py:729: UserWarning: [10:37:35] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
iteration: 100%|██████████| 90/90 [01:20<00:00,  1.12it/s]

===========   TRAIN RESULTS   ===========
confusion matrix of train :
predict\actual  Positive  Negative
      Positive   72090.0       0.0
      Negative       0.0  346032.0

metric of train :
     metric  Value
  Accuruacy    1.0
Sensitivity    1.0
Specificity    1.0
        PPV    1.0
        NPV    1.0
        MCC    1.0

===========   TEST RESULTS   ===========
confusion matrix of test :
predict\actual  Positive  Negative
      Positive      13.0      92.0
      Negative     797.0    3796.0

metric of test :
     metric   Value
  Accuruacy  0.8108
Sensitivity  0.0160
Specificity  0.9763
        PPV  0.1238
        NPV  0.8265
        MCC -0.0195



In [8]:
if not os.path.exists(save_combine_data_path):
    os.makedirs(save_combine_data_path)

# 定義 原始檔案前綴 與 期望輸出的檔名 對照表
file_mapping = {
    "y_train_label": "train_label.csv",
    "y_test_label": "test_label.csv",
    "yhat_train_probability": "train_prob.csv",
    "yhat_test_probability": "test_prob.csv"
}

print(f"正在合併檔案並存至: {save_combine_data_path}")

for prefix, output_name in file_mapping.items():
    # 搜尋所有 iteration 的檔案 (例如 y_train_label_0.csv, y_train_label_1.csv ...)
    search_pattern = os.path.join(save_data_path, f"{prefix}_*.csv")
    all_files = glob.glob(search_pattern)
    
    if all_files:
        # 讀取並合併
        df_list = [pd.read_csv(f) for f in all_files]
        combined_df = pd.concat(df_list, axis=0, ignore_index=True)
        
        # 依照 iteration 排序確保資料順序正確
        if 'iteration' in combined_df.columns:
            combined_df = combined_df.sort_values(by=['iteration']).reset_index(drop=True)
        
        # 儲存到指定路徑
        full_output_path = os.path.join(save_combine_data_path, output_name)
        combined_df.to_csv(full_output_path, index=False)
        print(f"✅ 已完成合併: {output_name}")
    else:
        print(f"❌ 找不到前綴為 {prefix} 的檔案，請檢查 save_data_path 是否正確。")

print("所有合併工作已結束。")

正在合併檔案並存至: D:\M143020071\xgboost_result\static_dynamic_ECGFounder_askna_mr_feature_result/combine/
✅ 已完成合併: train_label.csv
✅ 已完成合併: test_label.csv
✅ 已完成合併: train_prob.csv
✅ 已完成合併: test_prob.csv
所有合併工作已結束。
